# GeoLift-RT v2.1 — Colab train, validation và KITTI test

Notebook này chạy **GeoLift-S2 thật**: compact sparse prior, true-1/4 stems, stage-adapted MobileViTv2, PPG, RayLift-ID `K=(5,3,3,2)` và hard sparse anchor. Hai teacher archive được lấy đúng từ `docs/Install_dataset.md`.

Protocol dữ liệu: lấy giao ID giữa hai cache teacher → chọn cố định 1.000 mẫu → 800 train + 200 validation có GT. Bộ `test_1000` là KITTI anonymous chính thức, không có GT công khai nên chỉ xuất PNG để nộp benchmark; RMSE được báo trên `val_200`.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess

drive.mount("/content/drive")

REPO = Path(
    "/content/drive/MyDrive/DEPTH-FUSION | Workspace/"
    "monocular_sparse_fusion/GeoRT"
)

assert REPO.is_dir(), REPO
os.chdir(REPO)

subprocess.run(["git", "pull"], check=True)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

In [ ]:
# 1) Cấu hình duy nhất cần kiểm tra
from pathlib import Path

REPO_GIT_URL = ''  # điền URL nếu repo nằm trên Git
DRIVE_REPO_DIR = Path(
    "/content/drive/MyDrive/DEPTH-FUSION | Workspace/"
    "monocular_sparse_fusion/GeoRT"
)  # hoặc sửa path repo trên Drive
LOCAL_REPO = Path('/content/GeoDistill_RT')
DATA_ROOT = Path('/content/geolift_data')
RUN_LOCAL = Path('/content/geolift_run')
RUN_DRIVE = Path('/content/drive/MyDrive/GeoLift_RT_runs/v2_1_t2_1k')

METRIC_TEACHER_ID = '1ZtRVY67l3QkdgegSxDYtf6Cq_l-_BjQW'  # metric_coarse_train.tar, ~27 GB
DA_TEACHER_ID = '1DOtv8E_zW-pOkms2XUqro9vVfMnkqzbB'      # depth_anything_train_raw.tar, ~14 GB
SUBSET_COUNT, VAL_COUNT, SEED = 1000, 200, 42
SELECTION_STRATEGY = 'min_drives'  # giảm số raw-drive phải tải; dùng 'uniform' nếu đã có nhiều dung lượng/băng thông
EPOCHS, BATCH_SIZE, NUM_WORKERS = 30, 2, 2  # T4: bắt đầu batch 2; giảm 1 nếu OOM
assert SUBSET_COUNT == 1000 and VAL_COUNT == 200


In [ ]:
# 2) Mount Drive, stage source về SSD cục bộ và cài dependency gọn
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess

if LOCAL_REPO.exists():
    shutil.rmtree(LOCAL_REPO)
if REPO_GIT_URL:
    subprocess.run(['git', 'clone', '--depth=1', REPO_GIT_URL, str(LOCAL_REPO)], check=True)
else:
    assert DRIVE_REPO_DIR.is_dir(), f'Không thấy repo: {DRIVE_REPO_DIR}'
    subprocess.run(['rsync', '-a', '--exclude=.git/', '--exclude=venv/', '--exclude=data/',
                    '--exclude=teacher_outputs/', '--exclude=student_outputs/',
                    f'{DRIVE_REPO_DIR}/', f'{LOCAL_REPO}/'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '--upgrade',
                'gdown', 'pyyaml', 'timm', 'tqdm', 'pandas', 'scipy', 'opencv-python-headless'], check=True)
os.chdir(LOCAL_REPO)
import torch
assert torch.cuda.is_available(), 'Hãy chọn Runtime > Change runtime type > GPU'
print(torch.__version__, torch.cuda.get_device_name(0))
print(subprocess.run(['df', '-h', '/content'], text=True, capture_output=True, check=True).stdout)


## 3) Teacher 10k → giao ID → 1.000 mẫu

Hai tar cần khoảng 41 GB khi đang tải. Cell dừng nếu SSD không còn ít nhất 55 GB. Script chỉ giải nén 1.000 NPZ mỗi teacher và kiểm tra key; sau đó xóa tar để trả dung lượng. `min_drives` chỉ là subset kiểm thử kiến trúc, không được gọi là train protocol đầy đủ của paper.

In [ ]:
# 3a) Tải đúng hai teacher archive trong Install_dataset.md
import shutil, subprocess
DOWNLOADS = DATA_ROOT / 'downloads'
TEACHER_ROOT = DATA_ROOT / 'teacher_outputs'
SPLIT_ROOT = DATA_ROOT / 'splits'
for p in (DOWNLOADS, TEACHER_ROOT, SPLIT_ROOT, RUN_LOCAL, RUN_DRIVE):
    p.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage('/content').free / 1024**3
assert free_gb >= 55, f'Cần >=55 GB trống trước teacher download, hiện có {free_gb:.1f} GB'
metric_tar = DOWNLOADS / 'metric_coarse_train.tar'
da_tar = DOWNLOADS / 'depth_anything_train_raw.tar'
subprocess.run(['gdown', '--fuzzy', f'https://drive.google.com/file/d/{METRIC_TEACHER_ID}/view', '-O', str(metric_tar)], check=True)
subprocess.run(['gdown', '--fuzzy', f'https://drive.google.com/file/d/{DA_TEACHER_ID}/view', '-O', str(da_tar)], check=True)
assert metric_tar.stat().st_size > 20*1024**3 and da_tar.stat().st_size > 10*1024**3, 'Archive tải thiếu/hỏng'
print(metric_tar, metric_tar.stat().st_size/1024**3, 'GB')
print(da_tar, da_tar.stat().st_size/1024**3, 'GB')


In [ ]:
# 3b) Chọn giao teacher và giải nén đúng 800 train + 200 val
manifest = SPLIT_ROOT / 'teacher_subset_manifest.json'
subprocess.run([
    'python', 'scripts/extract_geolift_teachers.py',
    '--metric_tar', str(metric_tar), '--da_tar', str(da_tar),
    '--teacher_root', str(TEACHER_ROOT), '--manifest', str(manifest),
    '--count', str(SUBSET_COUNT), '--val_count', str(VAL_COUNT),
    '--seed', str(SEED), '--strategy', SELECTION_STRATEGY
], check=True)
metric_tar.unlink(); da_tar.unlink()
print(manifest.read_text()[:1200])
print('Free after teacher extraction:', shutil.disk_usage('/content').free/1024**3, 'GB')


## 4) Lấy input/GT và test 1.000 từ KITTI chính thức

Teacher cache không chứa RGB, sparse LiDAR, GT hay calibration. Ba ZIP dưới đây là nguồn KITTI chính thức. Script chỉ giải nén các frame được chọn và tự tải các raw RGB drive cần thiết; `data_depth_selection.zip` cung cấp đúng `test_depth_completion_anonymous` 1.000 ảnh.

In [ ]:
# 4a) Tải official depth archives (wget -c để resume)
KITTI_URL = 'https://s3.eu-central-1.amazonaws.com/avg-kitti'
official = {
    'annotated': ('data_depth_annotated.zip', f'{KITTI_URL}/data_depth_annotated.zip'),
    'velodyne': ('data_depth_velodyne.zip', f'{KITTI_URL}/data_depth_velodyne.zip'),
    'selection': ('data_depth_selection.zip', f'{KITTI_URL}/data_depth_selection.zip'),
}
for _, (name, url) in official.items():
    subprocess.run(['wget', '-q', '-c', url, '-O', str(DOWNLOADS/name)], check=True)
    print(name, (DOWNLOADS/name).stat().st_size/1024**3, 'GB')


In [ ]:
# 4b) Extract selective, tải raw RGB/calibration, tạo split chuẩn
KITTI_ROOT = DATA_ROOT / 'kitti'
subprocess.run([
    'python', 'scripts/extract_official_kitti_subset.py', '--manifest', str(manifest),
    '--annotated_zip', str(DOWNLOADS/official['annotated'][0]),
    '--velodyne_zip', str(DOWNLOADS/official['velodyne'][0]),
    '--selection_zip', str(DOWNLOADS/official['selection'][0]),
    '--data_root', str(KITTI_ROOT), '--split_root', str(SPLIT_ROOT),
    '--download_dir', str(DOWNLOADS)
], check=True)
for name, _ in official.values():
    (DOWNLOADS/name).unlink(missing_ok=True)
for name, expected in [('train_800.txt',800), ('val_200.txt',200), ('test_1000.txt',1000)]:
    count = sum(bool(x.strip()) for x in (SPLIT_ROOT/name).read_text().splitlines())
    assert count == expected, (name, count)
    print(name, count)


In [ ]:
# 5) Tạo resolved config; output train nằm local SSD, cuối mỗi pha sẽ sync Drive
import yaml
paths_file = DATA_ROOT / 'paths_geolift.yaml'
paths = {
    'data_root': str(KITTI_ROOT), 'split_root': str(SPLIT_ROOT),
    'train_split': 'train_800.txt', 'val_split': 'val_200.txt', 'test_split': 'test_1000.txt',
    'teacher_root': str(TEACHER_ROOT), 'student_root': str(RUN_LOCAL),
}
paths_file.write_text(yaml.safe_dump(paths, sort_keys=False))
cfg = yaml.safe_load((LOCAL_REPO/'configs/geolift_s2_v2_1.yaml').read_text())
cfg['paths_file'] = str(paths_file)
cfg['train']['epochs'] = EPOCHS; cfg['train']['batch_size'] = BATCH_SIZE
cfg['data']['num_workers'] = NUM_WORKERS
cfg['loss']['geometry_fallback'] = True  # baseline hai archive: cho phép DA raw fallback
cfg['teacher_checks']['require_geometry_fused'] = False
cfg['mono_ssi']['enabled'] = True
cfg['outputs']['backup_root'] = str(RUN_DRIVE)  # backup last/best + logs sau mỗi epoch
last_drive_ckpt = RUN_DRIVE/'checkpoints/last.pth'
if last_drive_ckpt.exists():
    (RUN_LOCAL/'checkpoints').mkdir(parents=True, exist_ok=True)
    shutil.copy2(last_drive_ckpt, RUN_LOCAL/'checkpoints/last.pth')
    cfg['train']['resume'] = str(RUN_LOCAL/'checkpoints/last.pth')
resolved_cfg = DATA_ROOT/'geolift_s2_colab.yaml'
resolved_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False))
shutil.copy2(resolved_cfg, RUN_LOCAL/'resolved_config.yaml')
shutil.copy2(paths_file, RUN_LOCAL/'resolved_paths.yaml')
shutil.copy2(manifest, RUN_LOCAL/'teacher_subset_manifest.json')
import hashlib, json
run_manifest = {
    'architecture': 'GeoLift-S2-v2.1', 'teacher_profile': 'T2 metric+DA',
    'seed': SEED, 'train_count': 800, 'val_count': 200, 'official_test_count': 1000,
    'selection_strategy': SELECTION_STRATEGY, 'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0),
    'git_commit': subprocess.run(['git','rev-parse','HEAD'], text=True, capture_output=True).stdout.strip() or 'drive-snapshot',
    'config_sha256': hashlib.sha256(resolved_cfg.read_bytes()).hexdigest(),
}
(RUN_LOCAL/'run_manifest.json').write_text(json.dumps(run_manifest, indent=2))
print(resolved_cfg.read_text())


In [ ]:
# 6) Bắt buộc: unit test + dataset/teacher/model preflight
subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)
from src.utils import load_project_config
from src.train_student import make_loader, preflight_teacher_coverage
from src.model_factory import build_student
import logging, torch
loaded_cfg, loaded_paths = load_project_config(resolved_cfg)
train_loader = make_loader(loaded_cfg, loaded_paths, 'train', training=True)
preflight_teacher_coverage(loaded_cfg, train_loader.dataset, logging.getLogger('preflight'))
batch = next(iter(train_loader))
model = build_student(loaded_cfg).cuda().eval()
with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
    out = model(*(batch[k][:1].cuda() for k in ('rgb','sparse','mask','ray','uv','K')))
assert out['D1'].shape[-2:] == tuple(loaded_cfg['data']['image_size'])
assert float(((out['D_full']-batch['sparse'][:1].cuda())*batch['mask'][:1].cuda()).abs().max()) == 0.0
print({k: tuple(out[k].shape) for k in ('D16','D8','D4','D2','D1','D_full')})
del model, out, batch; torch.cuda.empty_cache()


In [ ]:
# 7) Train/resume GeoLift-S2 T2
subprocess.run(['python', '-m', 'src.train_student', '--config', str(resolved_cfg)], check=True)
subprocess.run(['rsync', '-a', f'{RUN_LOCAL}/', f'{RUN_DRIVE}/'], check=True)
best_ckpt = RUN_LOCAL/'checkpoints/best.pth'
assert best_ckpt.exists(), best_ckpt


In [ ]:
# 8) Metric val_200 + RMSE tại từng stage; test_1000 xuất benchmark PNG
for split in ('val', 'test'):
    subprocess.run(['python', '-m', 'src.infer_student', '--config', str(resolved_cfg),
                    '--checkpoint', str(best_ckpt), '--split', split], check=True)
global_metric = RUN_LOCAL/'logs/infer_val_metrics_global.json'
print(global_metric.read_text())
test_png = list((RUN_LOCAL/'test_predictions/benchmark_png').glob('*.png'))
assert len(test_png) == 1000, len(test_png)
shutil.make_archive(str(RUN_LOCAL/'kitti_test_1000'), 'zip', RUN_LOCAL/'test_predictions/benchmark_png')
subprocess.run(['rsync', '-a', f'{RUN_LOCAL}/', f'{RUN_DRIVE}/'], check=True)
print('Official test PNG:', len(test_png), '->', RUN_DRIVE/'kitti_test_1000.zip')


In [ ]:
# 9) Runtime/component profile: batch=1, FP16, 352x1216, CUDA synchronize
profile_json = RUN_LOCAL/'logs/geolift_component_profile.json'
subprocess.run(['python', 'scripts/profile_geolift_s2.py', '--config', str(resolved_cfg),
                '--warmup', '30', '--runs', '100', '--output', str(profile_json)], check=True)
print(profile_json.read_text())
subprocess.run(['rsync', '-a', f'{RUN_LOCAL}/', f'{RUN_DRIVE}/'], check=True)


## Kết quả cần đọc

- `logs/infer_val_metrics_global.json`: RMSE/MAE/iRMSE/iMAE global và `stage_rmse_m` cho `D_init, D16, D8, D4, D2, D1, D_full`.
- `logs/geolift_component_profile.json`: median/P95 runtime, FPS, peak VRAM và runtime từng component.
- `kitti_test_1000.zip`: đúng 1.000 PNG uint16 (m × 256) để nộp KITTI; không có RMSE local vì test GT không công khai.
- Đây là profile teacher **T2** (metric + DA relative). `lambda_plane=0`; không ghi kết quả này là full T3/DSINE-RSGD.